# Install Requirements

In [2]:
!pip install playwright pandas
!playwright install


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


|                                                                                |   0% of 148.9 MiB
|■■■■■■■■                                                                        |  10% of 148.9 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

# Scraping (masih belum fix)

In [6]:
import asyncio
import json
# import time
from playwright.async_api import async_playwright

# 📍 Lokasi target (ID OLX)
LOCATIONS = {
    "jawa_timur": "2000011",
    "jawa_barat": "2000009",
    "jakarta_dki": "2000007",
    "banten": "2000004"
}

MAX_PAGES = 3
BASE_URL = "https://www.olx.co.id/dijual-rumah-apartemen_c5158"


async def scrape_olx_window_app(locations=LOCATIONS, max_pages=MAX_PAGES, base_url=BASE_URL):
    """
    Versi asinkron untuk dijalankan di Jupyter Notebook.
    Mengambil data window.__APP dari setiap halaman OLX per provinsi.
    """
    all_data = {}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for province, loc_id in locations.items():
            province_data = {}
            print(f"🔹 Memproses provinsi: {province}")

            for page_num in range(1, max_pages + 1):
                url = f"{base_url}?location={loc_id}&page={page_num}"
                print(f"  ➤ Mengambil halaman {page_num} dari {province}...")

                try:
                    await page.goto(url, timeout=60000)
                    await asyncio.sleep(2.5)  # tunggu JS render

                    app_data = await page.evaluate("window.__APP")
                    if app_data and "states" in app_data:
                        province_data[f"page_{page_num}"] = app_data
                        print("     ✅ Data window.__APP berhasil diambil.")
                    else:
                        print("     ⚠️ window.__APP kosong atau tidak ditemukan.")
                except Exception as e:
                    print(f"     ⚠️ Gagal memuat halaman {page_num} - {e}")

            # simpan hasil tiap provinsi ke file JSON
            filename = f"olx_{province}.json"
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(province_data, f, ensure_ascii=False, indent=2)
            all_data[province] = province_data

            print(f"✅ Selesai: {province}, total halaman tersimpan: {len(province_data)}\n")

        await browser.close()

    print("🎉 Semua provinsi selesai diekstraksi!")
    return all_data

In [7]:
data = await scrape_olx_window_app()

NotImplementedError: 